To be converted to scripts eventually?

Desired columns for LIB

- publisher ?
- title (from "title") 
- subject 1, subject 2, subject 3, (or more?), 
- se subjects
- series from "belongs-to-collection"?
- detective (add later) 
- collection type from "collection type"
- series position from "group-position"
- short description from "description"
- long description from "long-description"
- language from dc:language
- se word-count from "se:word-count"
- flesch scale from "se:reading-ease.flesch"
- source_url from "se:url.vcs.github"
- author from "author"
- author full name? from "se:name.person.full-name" (DON'T NEED)
- wikipedia url from "se:url.encyclopedia.wikipedia"
- number of chapters (DO LATER IN CORPUS)
- written_as (manually)
- work_type (manually)
- alt_titles (LATER) (manually)

Design decisions:
- choosing to treat each story in poirot investigates as its own document since the stories are the unit of composition (4/9/26)

## Raw Data Ingestion

The script below was run once to clone the corpus repositories from Standard Ebooks. 
It is preserved here for reproducibility. Do not re-run — raw_data/ is already populated.

```bash
#!/bin/bash
# Ingestion script for Agatha Christie Standard Ebooks Corpus

mkdir raw_data
cd raw_data

echo "Cloning repositories..."
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-at-the-vicarage.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_giants-bread.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-secret-adversary.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-on-the-links.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-man-in-the-brown-suit.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-big-four.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_poirot-investigates.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-mysterious-affair-at-styles.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-secret-of-chimneys.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-seven-dials-mystery.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-of-roger-ackroyd.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-mystery-of-the-blue-train.git

echo "Ingestion complete."
```

## Construct LIB table

In [39]:
# import libraries
import pandas as pd
import glob
import os
import re
import xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup

In [40]:
# # get list of all files in the raw_data folder
# root_dir = 'raw_data'
# file_list = []

# for root, dirs, files in os.walk(root_dir):
#     for file in files:
#         # Join the folder path and file name to get the full path
#         full_path = os.path.join(root, file)
#         file_list.append(full_path)

# file_list

In [41]:
# helper functions for if a field is missing in the content.opf file, return None instead of throwing an error
def get_metadata(root, property_val, NS):
    el = root.find(f'opf:metadata/opf:meta[@property="{property_val}"]', NS)
    return el.text if el is not None else None

def get_dc(root, tag, NS):
    el = root.find(f'opf:metadata/dc:{tag}', NS)
    return el.text if el is not None else None

In [42]:
# specify root directory (or hardcode?)
root_dir = 'raw_data'

# define namespaces used in StandardEbooks opf files (namespaces avoid tag collisions when multiple vocabularies are in one document)
NS = {
    'opf':  'http://www.idpf.org/2007/opf', # open packaging format
    'dc':   'http://purl.org/dc/elements/1.1/', # dublin core vocabulary (dublin core metadata terms)
    'rdf':  'http://www.w3.org/1999/02/22-rdf-syntax-ns#', # resource description framework
}

# get content.opf file paths
opf_files = sorted(Path(root_dir).glob('*/src/epub/content.opf'))

# initialize empty list to store book info and path
book_data = [] # a list of dictionaries, where each dictionary contains the book info for one book

for opf_path in opf_files:
    # get the book_id from the folder name
    book_id = opf_path.parts[1].replace('agatha-christie_', '') # .parts breaks a path into a tuple of components
    
    # initalize a list for subjects and se_subjects
    subjects = []
    se_subjects = []
    # get the tree and root for ElementTree parsing
    tree = ET.parse(opf_path)
    root = tree.getroot()

    # parse opf file and extract relevant fields
    publisher = get_dc(root, 'publisher', NS) # use helper function to guard against missing field
    title = get_dc(root, 'title', NS)
    series = get_metadata(root, 'belongs-to-collection', NS)
    col_type = get_metadata(root, 'collection-type', NS)
    series_pos = get_metadata(root, 'group-position', NS)
    short_desc = get_dc(root, 'description', NS)
    long_desc = get_metadata(root, 'se:long-description', NS)
    language = get_dc(root, 'language', NS)
    se_word_count = get_metadata(root, 'se:word-count', NS)
    se_reading_ease = get_metadata(root, 'se:reading-ease.flesch', NS)
    source_url = get_metadata(root, 'se:url.vcs.github', NS)
    author = get_dc(root, 'creator', NS)
    wikipedia_url = get_metadata(root, 'se:url.encyclopedia.wikipedia', NS)

    # for each subject in the content.opf file, append to subjects list
    subject_els = root.findall('opf:metadata/dc:subject', NS)
    subjects = '|'.join([el.text for el in subject_els]) if subject_els else None # join and separate subjects with pipe and guard against missing field

    se_subject_els = root.findall('opf:metadata/opf:meta[@property="se:subject"]', NS)
    se_subjects = '|'.join([el.text for el in se_subject_els]) if se_subject_els else None

    # append extracted fields to book_data list as a dictionary
    book_data.append({
        'book_id': book_id,
        'publisher': publisher,
        'title': title,
        'author': author,
        'subjects': subjects,
        'se_subjects': se_subjects,
        'col_type': col_type,
        'series': series,
        'series_pos': series_pos,
        'short_desc': short_desc,
        'long_desc': long_desc,
        'language': language,
        'se_word_count': se_word_count,
        'se_reading_ease': se_reading_ease,
        'source_url': source_url,
        'wikipedia_url': wikipedia_url,
    })

# build dataframe from book_data list and set book_id as index
LIB = pd.DataFrame(book_data).set_index('book_id')
LIB

,publisher,title,author,subjects,se_subjects,col_type,series,series_pos,short_desc,long_desc,language,se_word_count,se_reading_ease,source_url,wikipedia_url
book_id,,,,,,,,,,,,,,,
giants-bread,Standard Ebooks,Giant’s Bread,Agatha Christie,"Psychological fiction, English|Composers--Fiction",Fiction,NaN,NaN,NaN,The genesis of a monumental piece of music is ...,\n\t\t\t<p>Vernon Deyre lives with his parents...,en-GB,101165,78.45,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/Giant%27s_Bread
poirot-investigates,Standard Ebooks,Poirot Investigates,Agatha Christie,Private investigators -- England -- Fiction|De...,Mystery|Shorts,series,Poirot,3,A calculating Belgian detective and his wide-e...,\n\t\t\t<p><i>Poirot Investigates</i> is a cla...,en-GB,52707,76.72,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/Poirot_Investigates
the-big-four,Standard Ebooks,The Big Four,Agatha Christie,Private investigators -- England -- Fiction|De...,Mystery,series,Poirot,5,A famous detective must use all his little gre...,\n\t\t\t<p>“The American Soap King” has offere...,en-GB,55818,76.62,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Big_Four_(no...
the-man-in-the-brown-suit,Standard Ebooks,The Man in the Brown Suit,Agatha Christie,South Africa -- Fiction|Detective and mystery ...,Adventure|Fiction|Mystery,series,Colonel Race,1,Anne Beddingfeld travels to South Africa after...,"\n\t\t\t<p>After her father’s death, young Ann...",en-GB,75682,76.82,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Man_in_the_B...
the-murder-at-the-vicarage,Standard Ebooks,The Murder at the Vicarage,Agatha Christie,"Marple, Jane (Fictitious character)--Fiction|W...",Fiction|Mystery,series,Miss Marple,1,A vicar attempts to unravel the mystery of a m...,\n\t\t\t<p>Not much happens in the sleepy rura...,en-GB,70129,77.84,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Murder_at_th...
the-murder-of-roger-ackroyd,Standard Ebooks,The Murder of Roger Ackroyd,Agatha Christie,Private investigators -- England -- Fiction|De...,Mystery,series,Poirot,4,A legendary Belgian detective comes out of ret...,\n\t\t\t<p>Hercule Poirot has retired to the E...,en-GB,70105,77.84,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Murder_of_Ro...
the-murder-on-the-links,Standard Ebooks,The Murder on the Links,Agatha Christie,"Poirot, Hercule (Fictitious character)--Fictio...",Fiction|Mystery,series,Poirot,2,A famous Belgian detective and his sidekick ar...,\n\t\t\t<p><i>The Murder on the Links</i> is <...,en-GB,64890,77.33,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Murder_on_th...
the-mysterious-affair-at-styles,Standard Ebooks,The Mysterious Affair at Styles,Agatha Christie,"Poirot, Hercule (Fictitious character)--Fictio...",Fiction|Mystery,series,Poirot,1,A fastidious Belgian detective solves the myst...,\n\t\t\t<p><i>The Mysterious Affair at Styles<...,en-GB,56894,77.03,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Mysterious_A...
the-mystery-of-the-blue-train,Standard Ebooks,The Mystery of the Blue Train,Agatha Christie,Private investigators--England--Fiction|Railro...,Fiction|Mystery,series,Poirot,6,A retired detective finds himself at the scene...,\n\t\t\t<p><i>The Mystery of the Blue Train</i...,en-GB,70358,77.23,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Mystery_of_t...


In [43]:
# add manual fields
# add work type, writing_as, pub_year_original, genre, and sleuth by hand (from agathachristie.com and the agatha christie reading list app):
work_type = { # from the website
    'giants-bread': 'novel', # confirmed
    'poirot-investigates': 'collection of short stories', # confirmed
    'the-big-four': 'novel', # confirmed
    'the-man-in-the-brown-suit': 'novel', # confirmed
    'the-murder-at-the-vicarage': 'novel', # confirmed
    'the-murder-of-roger-ackroyd': 'novel', # confirmed
    'the-murder-on-the-links': 'novel', # confirmed
    'the-mysterious-affair-at-styles': 'novel', # confirmed
    'the-mystery-of-the-blue-train': 'novel', # confirmed
    'the-secret-adversary': 'novel', # confirmed
    'the-secret-of-chimneys': 'novel', # confirmed
    'the-seven-dials-mystery': 'novel' # confirmed
}

writing_as = { # from the website
    'giants-bread': 'Mary Westmacott', # confirmed
    'poirot-investigates': 'Agatha Christie', # confirmed
    'the-big-four': 'Agatha Christie', # confirmed
    'the-man-in-the-brown-suit': 'Agatha Christie', # confirmed
    'the-murder-at-the-vicarage': 'Agatha Christie', # confirmed
    'the-murder-of-roger-ackroyd': 'Agatha Christie', # confirmed
    'the-murder-on-the-links': 'Agatha Christie', # confirmed
    'the-mysterious-affair-at-styles': 'Agatha Christie', # confirmed
    'the-mystery-of-the-blue-train': 'Agatha Christie', # confirmed
    'the-secret-adversary': 'Agatha Christie', # confirmed
    'the-secret-of-chimneys': 'Agatha Christie', # confirmed
    'the-seven-dials-mystery': 'Agatha Christie' # confirmed
}

pub_year_original = { # from the website
    'giants-bread': 1930, # confirmed
    'poirot-investigates': 1924, # confirmed
    'the-big-four': 1927, # confirmed
    'the-man-in-the-brown-suit': 1924, # confirmed
    'the-murder-at-the-vicarage': 1930, # confirmed
    'the-murder-of-roger-ackroyd': 1926, # confirmed
    'the-murder-on-the-links': 1923, # confirmed
    'the-mysterious-affair-at-styles': 1920, # confirmed
    'the-mystery-of-the-blue-train': 1928, # confirmed
    'the-secret-adversary': 1922, # confirmed
    'the-secret-of-chimneys': 1925, # confirmed
    'the-seven-dials-mystery': 1929, # confirmed
}

genre = { # from the app
    'giants-bread': "Romance", # confirmed
    'poirot-investigates': "Murder Mystery|Detective|Thriller|Espionage", # confirmed
    'the-big-four': "Detective|Espionage|Thriller", # confirmed
    'the-man-in-the-brown-suit': "Murder Mystery|Thriller", # confirmed
    'the-murder-at-the-vicarage': "Murder Mystery", # confirmed
    'the-murder-of-roger-ackroyd': "Murder Mystery|Detective", # confirmed
    'the-murder-on-the-links': "Murder Mystery|Detective", # confirmed
    'the-mysterious-affair-at-styles': "Murder Mystery|Detective", # confirmed
    'the-mystery-of-the-blue-train': "Murder Mystery|Detective", # confirmed
    'the-secret-adversary': "Espionage|Thriller", # confirmed
    'the-secret-of-chimneys': "Espionage", # confirmed
    'the-seven-dials-mystery': "Espionage|Thriller", # confirmed
}

sleuth = { # from the app
    'giants-bread': None, # confirmed
    'poirot-investigates': "Hercule Poirot", # confirmed
    'the-big-four': "Hercule Poirot", # confirmed
    'the-man-in-the-brown-suit': "Colonel Race", # confirmed
    'the-murder-at-the-vicarage': "Miss Jane Marple", # confirmed
    'the-murder-of-roger-ackroyd': "Hercule Poirot", # confirmed
    'the-murder-on-the-links': "Hercule Poirot", # confirmed
    'the-mysterious-affair-at-styles': "Hercule Poirot", # confirmed
    'the-mystery-of-the-blue-train': "Hercule Poirot", # confirmed
    'the-secret-adversary': "Tommy and Tuppence", # confirmed
    'the-secret-of-chimneys': "Superintendent Battle", # confirmed
    'the-seven-dials-mystery': "Superintendent Battle", # confirmed
}

# join the dictionaries to LIB
LIB['pub_year_original'] = LIB.index.map(pub_year_original)
LIB['writing_as'] = LIB.index.map(writing_as)
LIB['work_type'] = LIB.index.map(work_type)
LIB['genre'] = LIB.index.map(genre)
LIB['sleuth'] = LIB.index.map(sleuth)

LIB


,publisher,title,author,subjects,se_subjects,col_type,series,series_pos,short_desc,long_desc,language,se_word_count,se_reading_ease,source_url,wikipedia_url,pub_year_original,writing_as,work_type,genre,sleuth
book_id,,,,,,,,,,,,,,,,,,,,
giants-bread,Standard Ebooks,Giant’s Bread,Agatha Christie,"Psychological fiction, English|Composers--Fiction",Fiction,NaN,NaN,NaN,The genesis of a monumental piece of music is ...,\n\t\t\t<p>Vernon Deyre lives with his parents...,en-GB,101165,78.45,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/Giant%27s_Bread,1930,Mary Westmacott,novel,Romance,NaN
poirot-investigates,Standard Ebooks,Poirot Investigates,Agatha Christie,Private investigators -- England -- Fiction|De...,Mystery|Shorts,series,Poirot,3,A calculating Belgian detective and his wide-e...,\n\t\t\t<p><i>Poirot Investigates</i> is a cla...,en-GB,52707,76.72,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/Poirot_Investigates,1924,Agatha Christie,collection of short stories,Murder Mystery|Detective|Thriller|Espionage,Hercule Poirot
the-big-four,Standard Ebooks,The Big Four,Agatha Christie,Private investigators -- England -- Fiction|De...,Mystery,series,Poirot,5,A famous detective must use all his little gre...,\n\t\t\t<p>“The American Soap King” has offere...,en-GB,55818,76.62,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Big_Four_(no...,1927,Agatha Christie,novel,Detective|Espionage|Thriller,Hercule Poirot
the-man-in-the-brown-suit,Standard Ebooks,The Man in the Brown Suit,Agatha Christie,South Africa -- Fiction|Detective and mystery ...,Adventure|Fiction|Mystery,series,Colonel Race,1,Anne Beddingfeld travels to South Africa after...,"\n\t\t\t<p>After her father’s death, young Ann...",en-GB,75682,76.82,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Man_in_the_B...,1924,Agatha Christie,novel,Murder Mystery|Thriller,Colonel Race
the-murder-at-the-vicarage,Standard Ebooks,The Murder at the Vicarage,Agatha Christie,"Marple, Jane (Fictitious character)--Fiction|W...",Fiction|Mystery,series,Miss Marple,1,A vicar attempts to unravel the mystery of a m...,\n\t\t\t<p>Not much happens in the sleepy rura...,en-GB,70129,77.84,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Murder_at_th...,1930,Agatha Christie,novel,Murder Mystery,Miss Jane Marple
the-murder-of-roger-ackroyd,Standard Ebooks,The Murder of Roger Ackroyd,Agatha Christie,Private investigators -- England -- Fiction|De...,Mystery,series,Poirot,4,A legendary Belgian detective comes out of ret...,\n\t\t\t<p>Hercule Poirot has retired to the E...,en-GB,70105,77.84,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Murder_of_Ro...,1926,Agatha Christie,novel,Murder Mystery|Detective,Hercule Poirot
the-murder-on-the-links,Standard Ebooks,The Murder on the Links,Agatha Christie,"Poirot, Hercule (Fictitious character)--Fictio...",Fiction|Mystery,series,Poirot,2,A famous Belgian detective and his sidekick ar...,\n\t\t\t<p><i>The Murder on the Links</i> is <...,en-GB,64890,77.33,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Murder_on_th...,1923,Agatha Christie,novel,Murder Mystery|Detective,Hercule Poirot
the-mysterious-affair-at-styles,Standard Ebooks,The Mysterious Affair at Styles,Agatha Christie,"Poirot, Hercule (Fictitious character)--Fictio...",Fiction|Mystery,series,Poirot,1,A fastidious Belgian detective solves the myst...,\n\t\t\t<p><i>The Mysterious Affair at Styles<...,en-GB,56894,77.03,https://github.com/standardebooks/agatha-chris...,https://en.wikipedia.org/wiki/The_Mysterious_A...,1920,Agatha Christie,novel,Murder Mystery|Detective,Hercule Poirot
the-mystery-of-the-blue-train,Standard Ebooks,The Mystery of the Blue Train,Agatha Christie,Private investigators--England--Fiction|Railro...,Fiction|Mystery,series,Poirot,6,A retired detective finds

In [44]:
# check for missingness
print(LIB.to_string())
print("\nMissing values:")
print(LIB.isnull().sum())

                                       publisher                            title           author                                                                                                                                                                                                                                                  subjects                se_subjects col_type                 series series_pos                                                                                                                                           short_desc                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [45]:
# fix data types
LIB['pub_year_original'] = LIB['pub_year_original'].astype('Int64')
LIB['se_word_count'] = LIB['se_word_count'].astype('Int64')
LIB['se_reading_ease'] = LIB['se_reading_ease'].astype('float64')

In [46]:
# save to csv

LIB.to_csv('data/LIB.csv', sep='\t', index=True)